# RND4Impact — full pipeline on Google Colab

Run **script_setup → image_setup → vid_setup** sequentially on a GPU runtime (no Docker).

| Stage | What it does | Output |
|-------|----------------|--------|
| **script_setup** | vLLM stages 1–4: ideas, title, scenes, image prompts | `data/<script_id>/script.json` |
| **image_setup** | SDXL generation + refinement | `data/<script_id>/refined_images/scene_XX.png` |
| **vid_setup** | Image-to-video (Wan I2V on T4 profile) | `data/<script_id>/raw_videos/scene_XX.mp4` |

**Before you start**
1. **Runtime → Change runtime type → T4 GPU** (or A100 for faster runs).
2. Optional: Colab **Secrets** → add `HF_TOKEN` for gated models / higher rate limits.
3. First install cell may take ~10–15 min. If Colab warns about a restart, run **Runtime → Restart session**, then re-run from **Parameters** onward (skip install if packages are already present).

Artifacts live under `data/` in the cloned repo (or on Google Drive if you enable persistence).

## 1. Parameters

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# --- edit these ---
REPO_URL = "https://github.com/VinSTyagi/RND4ImpactProject.git"  # or your fork
REPO_BRANCH = "main"
PROFILE = "t4"  # "t4" (16GB) | "low_vram" (~6GB Colab free tier)
SKIP_INSTALL = False  # set True after first successful install + restart
PREFETCH_MODELS = True  # download HF weights before inference
USE_GOOGLE_DRIVE = False  # persist data/ across sessions
DRIVE_SUBDIR = "rnd4impact"  # under MyDrive when USE_GOOGLE_DRIVE=True
FORCE_RECLONE = False  # set True to delete and re-clone the repo
REFRESH_T4_CONFIGS = False  # set True to overwrite stale notebook-written t4 YAMLs

PROFILES = {
    "t4": {
        "script": "configs/script_setup_t4.yaml",
        "image": "configs/image_setup_t4.yaml",
        "video": "configs/vid_setup_t4.yaml",
    },
    "low_vram": {
        "script": "configs/script_setup_phi4_mini.yaml",
        "image": "configs/image_setup_sdxl_low_vram.yaml",
        "video": "configs/vid_setup_svd_low_vram.yaml",
    },
}

if PROFILE not in PROFILES:
    raise ValueError(f"Unknown PROFILE={PROFILE!r}; choose from {list(PROFILES)}")

CONFIGS = PROFILES[PROFILE]
print(f"Using profile: {PROFILE}")
for stage, cfg in CONFIGS.items():
    print(f"  {stage}: {cfg}")


## 2. Clone repository & optional Drive mount

In [ ]:
import shutil

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK_ROOT = Path("/content/drive/MyDrive") / DRIVE_SUBDIR
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
else:
    WORK_ROOT = Path("/content")

REPO_ROOT = WORK_ROOT / "RND4ImpactProject"


def repo_is_ready(root: Path) -> bool:
    return (root / "src" / "script_setup" / "script_setup_runner.py").is_file()


def clone_repo() -> None:
    if "YOUR_ORG" in REPO_URL:
        raise ValueError(
            "Set REPO_URL in the Parameters cell to your GitHub repo "
            "(e.g. https://github.com/VinSTyagi/RND4ImpactProject.git)."
        )
    print(f"Cloning {REPO_URL} (branch {REPO_BRANCH}) -> {REPO_ROOT}")
    try:
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                REPO_BRANCH,
                "--depth",
                "1",
                REPO_URL,
                str(REPO_ROOT),
            ],
            check=True,
            capture_output=True,
            text=True,
        )
    except subprocess.CalledProcessError as exc:
        stderr = (exc.stderr or "").strip()
        raise RuntimeError(
            f"git clone failed (exit {exc.returncode}). "
            f"Check REPO_URL={REPO_URL!r} and REPO_BRANCH={REPO_BRANCH!r}.\n{stderr}"
        ) from exc


if FORCE_RECLONE and REPO_ROOT.exists():
    print(f"FORCE_RECLONE: removing {REPO_ROOT}")
    shutil.rmtree(REPO_ROOT)

if repo_is_ready(REPO_ROOT):
    print(f"Repo ready at {REPO_ROOT}")
elif (REPO_ROOT / ".git").is_dir():
    print(f"Updating existing git checkout at {REPO_ROOT}")
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", REPO_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH],
        check=True,
    )
    if not repo_is_ready(REPO_ROOT):
        print("Checkout still incomplete after pull; re-cloning.")
        shutil.rmtree(REPO_ROOT)
        clone_repo()
elif REPO_ROOT.exists():
    # Common when an earlier run created only data/ under REPO_ROOT.
    print(f"Removing incomplete folder {REPO_ROOT} (missing src/script_setup)")
    shutil.rmtree(REPO_ROOT)
    clone_repo()
else:
    clone_repo()

if not repo_is_ready(REPO_ROOT):
    raise RuntimeError(
        f"Clone finished but {REPO_ROOT / 'src/script_setup'} is missing. "
        f"Check REPO_URL and REPO_BRANCH."
    )

DATA_DIR = REPO_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# t4 profile YAMLs may not be on GitHub main yet — materialize them after clone.
T4_CONFIG_FILES: dict[str, tuple[str, str]] = {
    "script_setup": (
        "configs/script_setup_t4.yaml",
        """# ~16GB VRAM (NVIDIA T4): Qwen3-8B-AWQ ~5GB weights + ~4GB KV @ 8192 ctx.
global_vllm_config:
  model_path: Qwen/Qwen3-8B-AWQ
  quantization: awq
  max_tokens: 6144
  temperature: 0.6
  top_p: 0.95
  top_k: 20
  min_p: 0
  max_model_len: 8192
  tensor_parallel_size: 1
  gpu_memory_utilization: 0.80
  enforce_eager: false
  enable_thinking: true
  max_num_seqs: 2
  max_num_batched_tokens: 8192

idea_config:
  num_ideas: 3
  prompt_path: script_setup/prompts/stage_1.md
  output_path: data/

title_config:
  num_words: 5
  prompt_path: script_setup/prompts/stage_2.md
  script_path: data/

scene_config:
  num_scenes: 7
  prompt_path: script_setup/prompts/stage_3.md
  script_path: data/

image_config:
  prompt_path: script_setup/prompts/stage_4.md
  script_path: data/
""",
    ),
    "image_setup": (
        "configs/image_setup_t4.yaml",
        """# ~16GB VRAM (NVIDIA T4): full SDXL base + refiner at 1344×768 via model offload.
pipeline_config:
  type: sdxl
  model_path: stabilityai/stable-diffusion-xl-base-1.0
  variant: fp16
  scheduler: euler

quantization_config:
  torch_dtype: float16
  enable_model_cpu_offload: true
  enable_sequential_cpu_offload: false
  enable_vae_slicing: true
  enable_vae_tiling: true
  enable_attention_slicing: false
  enable_xformers: false

generation_config:
  num_inference_steps: 35
  default_guidance_scale: 7.5
  seed: null
  device: cuda
  width: 1344
  height: 768

refinement_config:
  enabled: true
  type: sdxl_refiner
  model_path: stabilityai/stable-diffusion-xl-refiner-1.0
  variant: fp16
  scheduler: euler
  num_inference_steps: 25
  denoising_start: 0.8
  denoising_end: 0.8
  strength: 0.35

output_config:
  script_path: data/
  raw_subdir: raw_images
  output_subdir: refined_images
  filename_template: "scene_{scene_number:02d}.png"
  save_raw: true
  skip_existing: true
""",
    ),
    "vid_setup": (
        "configs/vid_setup_t4.yaml",
        """# ~16GB VRAM (NVIDIA T4): Wan 2.1 I2V 14B (best quality in repo) via model offload.
video_diffuser_config:
  type: wan
  model_path: Wan-AI/Wan2.1-I2V-14B-480P-Diffusers
  variant: null
  device: cuda

quantization_config:
  torch_dtype: bfloat16
  enable_sequential_cpu_offload: false
  enable_model_cpu_offload: true
  enable_vae_slicing: true
  enable_vae_tiling: true
  enable_attention_slicing: false
  enable_xformers: false

generation_config:
  num_inference_steps: 30
  num_frames: 17
  width: 512
  height: 288
  frame_rate: 8
  guidance_scale: 5.0
  prompt: "cinematic motion, smooth natural movement, high quality"
  negative_prompt: "worst quality, inconsistent motion, blurry, jittery, distorted, watermark, text"

io_config:
  script_path: data/
  input_subdir: refined_images
  image_template: "scene_{scene_number:02d}.png"
  raw_videos_subdir: raw_videos
  refined_videos_subdir: refined_videos
  video_template: "scene_{scene_number:02d}.mp4"
  save_raw: true
  skip_existing: true

upscale_config:
  enabled: false
  type: none
""",
    ),
}


def _git_tracks(path: Path) -> bool:
    rel = path.relative_to(REPO_ROOT).as_posix()
    result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "ls-files", "--error-unmatch", rel],
        capture_output=True,
    )
    return result.returncode == 0


def _read_git_config(rel_posix: str) -> str | None:
    result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "show", f"origin/{REPO_BRANCH}:{rel_posix}"],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        return result.stdout
    return None


def _summarize_script_quantization(config_path: Path) -> str:
    for line in config_path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if stripped.startswith("quantization:"):
            return stripped.split(":", 1)[1].strip()
    return "(not set)"


def ensure_profile_configs() -> None:
    setup_names = {
        "script": "script_setup",
        "image": "image_setup",
        "video": "vid_setup",
    }
    for stage, setup_name in setup_names.items():
        config_rel = CONFIGS[stage]
        setup_dir = REPO_ROOT / "src" / setup_name
        config_path = setup_dir / config_rel
        rel_posix = config_path.relative_to(REPO_ROOT).as_posix()

        if REFRESH_T4_CONFIGS and PROFILE == "t4" and setup_name in T4_CONFIG_FILES:
            bundled_rel, content = T4_CONFIG_FILES[setup_name]
            if bundled_rel != config_rel:
                raise ValueError(
                    f"Notebook t4 bundle mismatch for {setup_name}: "
                    f"expected {config_rel!r}, have {bundled_rel!r}"
                )
            config_path.parent.mkdir(parents=True, exist_ok=True)
            config_path.write_text(content, encoding="utf-8")
            print(f"Refreshed t4 config: {rel_posix}")
            if setup_name == "script_setup":
                print(f"  quantization: {_summarize_script_quantization(config_path)}")
            continue

        repo_content = _read_git_config(rel_posix)
        if repo_content is not None:
            if not config_path.is_file() or config_path.read_text(encoding="utf-8") != repo_content:
                config_path.parent.mkdir(parents=True, exist_ok=True)
                config_path.write_text(repo_content, encoding="utf-8")
                print(f"Synced config from origin/{REPO_BRANCH}: {rel_posix}")
            else:
                print(f"Config OK (repo): {rel_posix}")
            if setup_name == "script_setup":
                print(f"  quantization: {_summarize_script_quantization(config_path)}")
            continue

        if config_path.is_file():
            source = "repo" if _git_tracks(config_path) else "local/notebook"
            print(f"Config OK ({source}): {rel_posix}")
            if setup_name == "script_setup":
                print(f"  quantization: {_summarize_script_quantization(config_path)}")
            continue

        if PROFILE == "t4" and setup_name in T4_CONFIG_FILES:
            bundled_rel, content = T4_CONFIG_FILES[setup_name]
            if bundled_rel != config_rel:
                raise ValueError(
                    f"Notebook t4 bundle mismatch for {setup_name}: "
                    f"expected {config_rel!r}, have {bundled_rel!r}"
                )
            config_path.parent.mkdir(parents=True, exist_ok=True)
            config_path.write_text(content, encoding="utf-8")
            print(f"Wrote missing t4 config: {rel_posix}")
            if setup_name == "script_setup":
                print(f"  quantization: {_summarize_script_quantization(config_path)}")
            continue

        raise FileNotFoundError(
            f"Missing config for PROFILE={PROFILE!r}: {config_path}. "
            "Push the t4 YAML files to GitHub or switch PROFILE to 'low_vram'."
        )


ensure_profile_configs()


def patch_script_setup_runner_lazy_imports() -> None:
    """GitHub main may still eager-import vLLM; patch in place for Colab."""
    runner = REPO_ROOT / "src" / "script_setup" / "script_setup_runner.py"
    text = runner.read_text(encoding="utf-8")
    if "from utils import prefetch, stage_1, stage_2, stage_3, stage_4, vllm_wrapper" not in text:
        return
    text = text.replace(
        "from utils import prefetch, stage_1, stage_2, stage_3, stage_4, vllm_wrapper\n",
        "",
    )
    for n in (1, 2, 3, 4):
        needle = (
            f"def run_stage_{n}(pipeline_config: PipelineConfig, state: dict) -> dict:\n"
            f"    vcfg ="
        )
        repl = (
            f"def run_stage_{n}(pipeline_config: PipelineConfig, state: dict) -> dict:\n"
            f"    from utils import stage_{n}, vllm_wrapper\n\n"
            f"    vcfg ="
        )
        text = text.replace(needle, repl)
    text = text.replace(
        "    if args.prefetch:\n        model_paths = prefetch.collect_model_paths",
        "    if args.prefetch:\n        from utils import prefetch\n\n        model_paths = prefetch.collect_model_paths",
    )
    runner.write_text(text, encoding="utf-8")
    print("Patched script_setup_runner.py (lazy vLLM imports).")


patch_script_setup_runner_lazy_imports()

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR:", DATA_DIR)


## 3. Install dependencies (pip, no Docker)

Pins match the lean `pyproject.toml` files in each setup. Colab is Linux, so vLLM CUDA wheels install correctly.

In [ ]:
PYTORCH_INDEX = "https://download.pytorch.org/whl/cu124"

PIP_PACKAGES = [
    "torch==2.6.0",
    "xformers==0.0.29.post2",
    "vllm==0.8.5",
    "diffusers>=0.32,<0.40",
    "accelerate>=1.0",
    "safetensors>=0.4",
    "transformers>=4.51.1,<5",
    "huggingface-hub>=0.24",
    "pillow>=10.2",
    "pyyaml>=6.0",
    "tqdm>=4.66",
    "imageio>=2.34",
    "imageio-ffmpeg>=0.5",
]

# Colab pre-installs TensorFlow; it needs protobuf>=5. Other wheels may pin older protobuf.
POST_INSTALL_PACKAGES = [
    "protobuf>=5.26.1",
]

if not SKIP_INSTALL:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *PIP_PACKAGES,
            "--index-url",
            PYTORCH_INDEX,
            "--extra-index-url",
            "https://pypi.org/simple",
        ],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            *POST_INSTALL_PACKAGES,
        ],
        check=True,
    )
    print("Install finished. If Colab prompts to restart, do so and set SKIP_INSTALL=True.")
else:
    print("Skipping install (SKIP_INSTALL=True)")


## 4. Hugging Face token & GPU check

In [ ]:
import torch


def repair_colab_python_deps() -> None:
    """Fix protobuf vs TensorFlow mismatch after pip installs on Colab."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "protobuf>=5.26.1"],
        check=True,
    )
    import google.protobuf

    print(f"protobuf {google.protobuf.__version__}")


repair_colab_python_deps()

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if IN_COLAB:
    try:
        from google.colab import userdata

        HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
    except Exception:
        pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF token configured.")
else:
    print("No HF token — public models only; rate limits may apply.")

for key, value in {
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "VLLM_USE_V1": "0",
    "USE_TF": "0",
    "USE_TORCH": "1",
    "TF_CPP_MIN_LOG_LEVEL": "3",
}.items():
    os.environ.setdefault(key, value)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available — switch Runtime to GPU (T4 or better).")

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024**3)
print(f"GPU: {props.name} | {vram_gb:.1f} GB VRAM | torch {torch.__version__} | cuda {torch.version.cuda}")

if PROFILE == "t4" and vram_gb < 14:
    print("WARNING: PROFILE='t4' expects ~16GB. Consider PROFILE='low_vram' on smaller GPUs.")


## 5. Helpers

Each stage runs in a **subprocess** so vLLM fully releases GPU memory before diffusers loads.

In [ ]:
def run_setup(
    setup_name: str,
    runner: str,
    config: str,
    extra_args: list[str] | None = None,
) -> None:
    setup_dir = REPO_ROOT / "src" / setup_name
    config_path = setup_dir / config
    if not config_path.is_file():
        raise FileNotFoundError(
            f"Config not found: {config_path}. Re-run the clone cell (section 2)."
        )
    cmd = [sys.executable, runner, "--config", config, *(extra_args or ["--all"])]
    print("$", " ".join(cmd), f"(cwd={setup_dir})")
    env = os.environ.copy()
    result = subprocess.run(
        cmd,
        cwd=setup_dir,
        env=env,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    if result.returncode != 0:
        tail = (result.stderr or result.stdout or "").strip().splitlines()
        detail = "\n".join(tail[-20:]) if tail else "(no subprocess output captured)"
        raise RuntimeError(
            f"{setup_name} failed with exit code {result.returncode}.\n"
            f"Last output lines:\n{detail}"
        )


def list_script_runs() -> list[Path]:
    if not DATA_DIR.is_dir():
        return []
    return sorted(
        p for p in DATA_DIR.iterdir() if p.is_dir() and (p / "script.json").is_file()
    )


def latest_script_run() -> Path | None:
    runs = list_script_runs()
    return runs[-1] if runs else None


## 6. Prefetch models (optional, saves time on first inference)

In [ ]:
if PREFETCH_MODELS:
    run_setup(
        "script_setup",
        "script_setup_runner.py",
        CONFIGS["script"],
        extra_args=["--prefetch"],
    )
    print("script_setup weights prefetched.")
else:
    print("Skipping prefetch.")


## 7. Stage A — script_setup (vLLM, stages 1–4)

In [ ]:
run_setup("script_setup", "script_setup_runner.py", CONFIGS["script"])

run = latest_script_run()
if run is None:
    raise RuntimeError("No script.json found under data/ — script_setup did not produce output.")
print(f"Script written: {run / 'script.json'}")


## 8. Stage B — image_setup (SDXL raw + refine)

In [ ]:
run_setup("image_setup", "image_setup_runner.py", CONFIGS["image"])

run = latest_script_run()
images = sorted((run / "refined_images").glob("scene_*.png")) if run else []
print(f"Refined images: {len(images)}")
for p in images[:3]:
    print(" ", p.name)


## 9. Stage C — vid_setup (image → video)

In [ ]:
run_setup("vid_setup", "vid_setup_runner.py", CONFIGS["video"])

run = latest_script_run()
videos = sorted((run / "raw_videos").glob("scene_*.mp4")) if run else []
print(f"Raw videos: {len(videos)}")
for p in videos[:3]:
    print(" ", p.name)


## 10. Preview outputs

In [ ]:
from IPython.display import Image as IPyImage, Video, display

run = latest_script_run()
if run is None:
    print("No run found.")
else:
    print(f"Run folder: {run}")
    for img in sorted((run / "refined_images").glob("scene_*.png"))[:2]:
        display(IPyImage(filename=str(img)))
    for vid in sorted((run / "raw_videos").glob("scene_*.mp4"))[:1]:
        display(Video(str(vid), embed=True, width=640))


## Troubleshooting

| Issue | Fix |
|-------|-----|
| CUDA OOM on script | Use `PROFILE = "low_vram"` or lower `gpu_memory_utilization` / `max_model_len` in the YAML |
| CUDA OOM on image | Use `image_setup_sdxl_low_vram.yaml` or reduce width/height in config |
| CUDA OOM on video | Switch video config to `vid_setup_svd_low_vram.yaml` or enable CPU offload in YAML |
| Wrong `quantization` (e.g. stuck on `awq_marlin`) | Re-run section 2 — it syncs YAML from GitHub; or set `REFRESH_T4_CONFIGS = True` |
| vLLM import error | Ensure GPU runtime (Linux); re-run install + section 4 |
| Stale GPU memory | **Runtime → Restart**, then run stages individually with `SKIP_INSTALL=True` |
| Resume mid-pipeline | Set configs' `skip_existing: true` (default) and re-run only the stage you need |